# ShiftGuard-X NB60: Integrated_Evidence_and_Novelty_Gate

Loads earlier outputs from uploaded ZIPs, rejects missing or unsupported evidence, and records component-level overlap with ALP and PseudoDecimals. A potential future mechanism is explicitly marked unimplemented and unverified.

Run **Runtime → Run all** in a fresh Google Colab CPU session. The final cell writes a results ZIP and attempts a browser download. This work is exploratory and does not establish novelty, journal acceptance, or ALP superiority.


In [ ]:

import csv,io,json,zipfile,platform,traceback
from pathlib import Path
ROOT=Path('/content' if Path('/content').exists() else '/mnt/data')
OUT=ROOT/'SGX_NB60_outputs';OUT.mkdir(exist_ok=True,parents=True)
EXPECTED={'NB55':'SGX_NB55_results.zip','NB58':'SGX_NB58_results.zip','NB59':'SGX_NB59_results.zip'}
# Upload archives to /content if working in separate Colab runtimes. Missing inputs are not assumed successful.
missing=[name for name in EXPECTED.values() if not (ROOT/name).exists()]
if missing:
 print('Need result ZIPs:',missing)
 try:
  from google.colab import files
  print('Upload the missing ZIPs now (NB55 older result, NB58, and NB59).')
  files.upload()
 except ImportError: pass
excepted={}
def load_zip(tag,path):
 if not path.exists():return None
 try:
  with zipfile.ZipFile(path) as z:
   names=z.namelist()
   pick='decision.json' if tag in ('NB55','NB58') else 'status.json'
   if pick not in names:return {'read_error':'Missing '+pick,'files':names}
   return json.loads(z.read(pick))
 except Exception as exc:return {'read_error':repr(exc)}
res={tag:load_zip(tag,ROOT/path) for tag,path in EXPECTED.items()}
for tag,r in res.items():
 (OUT/(tag+'_status_snapshot.json')).write_text(json.dumps(r,indent=2,default=str))
novelty_rows=[
 {'component':'Decimal scaling float64->integers','existing_work':'PseudoDecimals; ALP','evidence':'ALP SIGMOD 2024, DOI 10.1145/3626717','novelty':'NOT ESTABLISHED'},
 {'component':'Raw-bit exception streams','existing_work':'ALP; PseudoDecimals; patched integer codecs','evidence':'ALP SIGMOD 2024','novelty':'NOT ESTABLISHED'},
 {'component':'Bit-packing / frame of reference','existing_work':'FastPFOR; ALP; integer compression','evidence':'ALP SIGMOD 2024','novelty':'NOT ESTABLISHED'},
 {'component':'Calibration-based fallback selector','existing_work':'ALP rowgroup/vector sampling; selector literature','evidence':'ALP SIGMOD 2024','novelty':'NOT ESTABLISHED'},
 {'component':'Potential cross-column joint residual coding','existing_work':'Column-group dictionaries and multivariate compressors require new review','evidence':'OPEN RESEARCH QUESTION; not implemented','novelty':'UNVERIFIED HYPOTHESIS'},
]
with (OUT/'prior_art_and_hypothesis.csv').open('w',newline='') as f:
 w=csv.DictWriter(f,fieldnames=list(novelty_rows[0]));w.writeheader();w.writerows(novelty_rows)
gates={
 'nb55_available':res['NB55'] is not None,
 'nb58_available':res['NB58'] is not None,
 'nb59_available':res['NB59'] is not None,
 'nb58_correctly_finished':res['NB58'] is not None and res['NB58'].get('state')=='COMPLETE',
 'nb58_all_expected_streams':res['NB58'] is not None and res['NB58'].get('streams_completed')==54,
 'nb59_official_alp_executed':res['NB59'] is not None and res['NB59'].get('stage')=='ALP_NATIVE_BENCHMARK_PASSED',
 'matched_alp_full_container_and_bit_exact_decode':False,
 'original_contribution_validated':False,
}
# Official ALP bench_your_dataset exports native bits/value, not compatible full-file bytes by itself.
# It does not verify a bit-exact independently roundtripped stream; no publication superiority can be inferred.
if not res['NB58'] or not res['NB59']:decision='INCOMPLETE_INPUTS'
elif not gates['nb58_correctly_finished']:decision='REPAIR_EXPERIMENT'
elif not gates['nb59_official_alp_executed']:decision='ALP_INTERFACE_REPAIR_REQUIRED'
else:decision='ORIGINAL_MECHANISM_AND_FAIR_NATIVE_MATCH_REQUIRED'
report={'decision':decision,'checks':gates,'input_files':EXPECTED,
        'nb58_size_saving_pct':res['NB58'].get('savings_vs_calibrated_baseline_percent') if res['NB58'] else None,
        'scope':'ALP native BPV is not compared against SGX complete-container bits/value; no unverified speed or compression claim.',
        'next_hypothesis':'Explore cross-column shared predictor with decoder-synchronous references only after a formal prior-art search; design matched ablations and run on unseen datasets.',
        'sources':['https://doi.org/10.1145/3626717',
                   'https://github.com/cwida/ALP',
                   'https://github.com/cwida/ALP/blob/main/how_to_benchmark_your_dataset.md']}
(OUT/'research_decision.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))
zip_path=ROOT/'SGX_NB60_results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
 for p in OUT.rglob('*'):
  if p.is_file():z.write(p,p.relative_to(OUT))
print('RESULTS ZIP:',zip_path)
try:
 from google.colab import files;files.download(str(zip_path))
except ImportError:pass
